# Embeddings Playground — coding workshop (advanced)

**Expected time:** ~15-20 minutes, individual, no GPU needed (both parts run fine on
CPU; the downloads take a minute or two).

Toy 2-D numbers — aunt = (4,2), uncle = (3,3), car = (1,4) — showed how cosine similarity
captures meaning. Here you compute those same three cosine similarities with *real*,
pretrained embeddings, then ask a sharper question: in a real multilingual model, do
Galician words actually look distinct, or do they blur into Spanish and Portuguese?

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lecture or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

## Part A — Real embeddings, real cosine similarity

`gensim` ships a simple downloader for small pretrained word-vector sets. We'll use
`glove-wiki-gigaword-50` — 50-dimensional vectors trained on Wikipedia text, about 66MB,
small enough to download and load in well under a minute.

In [ ]:
!pip install -q gensim

import gensim.downloader as api

print("Downloading glove-wiki-gigaword-50 (about 66MB)...")
word_vectors = api.load("glove-wiki-gigaword-50")
print("Loaded. Vocabulary size:", len(word_vectors))

## Compare to the toy numbers

Aunt = (4,2), uncle = (3,3), car = (1,4) gave cos(aunt, uncle) ≈ 0.95 and cos(aunt, car) ≈
0.65 — illustrative numbers, chosen to make the arithmetic clean. Real embeddings weren't
hand-picked for that. The cells below show how close the real numbers actually land.

### ✏️ Exercise 1 — Cosine similarity by hand

Write `cosine(v1, v2)`, then use it on the real word vectors. The cell prints your
value next to gensim's, and they should agree.

cos(v1, v2) = (v1 · v2) / (|v1| |v2|)

<details><summary><b>Hints</b> (click to open)</summary>

- `np.dot(a, b)` is the dot product; `np.linalg.norm(a)` is the length of `a`.
- `word_vectors["aunt"]` is the 50-dimensional vector for *aunt*.

</details>

In [ ]:
import numpy as np


def cosine(v1, v2):
    # TODO: cosine similarity of two vectors
    ___


for w1, w2 in [("aunt", "uncle"), ("aunt", "car"), ("uncle", "car")]:
    sim = ___  # TODO: your cosine of the two words' vectors
    print(f"cos({w1}, {w2}) = {sim:.3f}   (gensim: {word_vectors.similarity(w1, w2):.3f})")

In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(cosine(np.array([4, 2]), np.array([3, 3])) - 0.949) < 1e-3, "lecture toy example: cos(aunt, uncle) = 0.95"
assert abs(cosine(np.array([4, 2]), np.array([1, 4])) - 0.651) < 1e-3, "lecture toy example: cos(aunt, car) = 0.65"
assert abs(cosine(word_vectors["aunt"], word_vectors["uncle"]) - word_vectors.similarity("aunt", "uncle")) < 1e-5
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
import numpy as np


def cosine(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


for w1, w2 in [("aunt", "uncle"), ("aunt", "car"), ("uncle", "car")]:
    sim = cosine(word_vectors[w1], word_vectors[w2])
    print(f"cos({w1}, {w2}) = {sim:.3f}   (gensim: {word_vectors.similarity(w1, w2):.3f})")
```

</details>

## Nearest neighbors

What is this model's own notion of "close to aunt"?

### ✏️ Exercise 2 — Nearest neighbors by hand

`word_vectors.most_similar` is one line of code. Here you compute the same list
yourself: the cosine similarity of *aunt* to **every** word in the vocabulary at once, as one
matrix operation, then the 8 most similar words.

<details><summary><b>Hints</b> (click to open)</summary>

- `all_vectors @ target` gives one dot product per row.
- `np.linalg.norm(all_vectors, axis=1)` gives one length per row.
- `np.argsort` sorts from smallest to largest. Sorting `-sims` puts the largest first.
- `word_vectors.index_to_key[i]` is the word in row `i`.

</details>

In [ ]:
target = word_vectors["aunt"]
all_vectors = word_vectors.vectors  # one row per vocabulary word
sims = ___  # TODO: cosine similarity of target to every row of all_vectors
best_first = ___  # TODO: row indices sorted from most to least similar
my_neighbors = ___  # TODO: the 8 most similar words, excluding "aunt" itself

for word, score in word_vectors.most_similar("aunt", topn=8):
    print(f"{word:12s} {score:.3f}")
print("\nYours:", my_neighbors)

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert len(sims) == len(word_vectors), "one similarity per vocabulary word"
assert my_neighbors == [w for w, _ in word_vectors.most_similar("aunt", topn=8)], my_neighbors
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
target = word_vectors["aunt"]
all_vectors = word_vectors.vectors
sims = (all_vectors @ target) / (np.linalg.norm(all_vectors, axis=1) * np.linalg.norm(target))
best_first = np.argsort(-sims)
my_neighbors = [word_vectors.index_to_key[i] for i in best_first[:9]
                if word_vectors.index_to_key[i] != "aunt"][:8]

for word, score in word_vectors.most_similar("aunt", topn=8):
    print(f"{word:12s} {score:.3f}")
print("\nYours:", my_neighbors)
```

</details>

### ✏️ Exercise 3 — Normalize once, query many words at once (advanced)

Search engines and retrieval systems answer many similarity queries over the same
vectors. Two steps make that cheap:

1. **Normalize once.** Divide every vector by its length a single time. After that, the
   cosine of two vectors is just their dot product.
2. **Batch the queries.** Put the query vectors in one matrix Q, one row per word. Then
   Q · Uᵀ, with U the matrix of normalized vectors, gives every query's similarity to every
   vocabulary word in one matrix product.

To find the top k of 400,000 values, sorting all of them is wasted work.
`np.argpartition(-row, k)` puts the k largest first, in no particular order, without
sorting the rest; then sort just those k.

Complete the two functions. The cell times your version against calling
`most_similar` once per word, and checks that the neighbors agree. The query word itself is
always its own nearest neighbor; leave it out.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.linalg.norm(matrix, axis=1, keepdims=True)` has shape (rows, 1), so the division works row by row.
- `unit_vectors[rows]` picks the query rows; `@ unit_vectors.T` multiplies them with every vocabulary vector.
- `np.argpartition(-row, k)[:k]`: the k positions with the largest values of `row`.

</details>

In [ ]:
import time


def normalize_rows(matrix):
    # TODO: divide each row by its length (keep the shape: use keepdims=True)
    ___


def batch_neighbors(words, unit_vectors, k=5):
    rows = [word_vectors.key_to_index[w] for w in words]
    sims = ___  # TODO: one matrix product: (len(words), vocabulary size)
    neighbors = {}
    for word, row_index, row in zip(words, rows, sims):
        row[row_index] = -np.inf  # leave the word itself out
        top = ___  # TODO: indices of the k largest values, with np.argpartition
        top = top[np.argsort(-row[top])]  # sort just those k
        neighbors[word] = [word_vectors.index_to_key[i] for i in top]
    return neighbors

QUERY_WORDS = ["aunt", "uncle", "car", "bus", "bread", "cheese", "river", "city", "music", "doctor",
               "green", "energy", "computer", "language", "translation", "galicia", "spain", "ocean",
               "summer", "winter"]

start = time.time()
slow = {w: [n for n, _ in word_vectors.most_similar(w, topn=5)] for w in QUERY_WORDS}
elapsed_loop = time.time() - start

unit_vectors = normalize_rows(word_vectors.vectors)
start = time.time()
fast = batch_neighbors(QUERY_WORDS, unit_vectors, k=5)
elapsed_batch = time.time() - start

print(f"one query at a time: {elapsed_loop * 1000:7.1f} ms")
print(f"all queries at once: {elapsed_batch * 1000:7.1f} ms")
print(f"same neighbors: {sum(slow[w] == fast[w] for w in QUERY_WORDS)}/{len(QUERY_WORDS)}")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert unit_vectors.shape == word_vectors.vectors.shape
assert np.allclose(np.linalg.norm(unit_vectors[:100], axis=1), 1, atol=1e-5), "every row should have length 1"
assert sum(slow[w] == fast[w] for w in QUERY_WORDS) >= len(QUERY_WORDS) - 1, \
    "your neighbors should match most_similar (a near tie may swap one pair)"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import time


def normalize_rows(matrix):
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


def batch_neighbors(words, unit_vectors, k=5):
    rows = [word_vectors.key_to_index[w] for w in words]
    sims = unit_vectors[rows] @ unit_vectors.T
    neighbors = {}
    for word, row_index, row in zip(words, rows, sims):
        row[row_index] = -np.inf  # leave the word itself out
        top = np.argpartition(-row, k)[:k]
        top = top[np.argsort(-row[top])]  # sort just those k
        neighbors[word] = [word_vectors.index_to_key[i] for i in top]
    return neighbors

QUERY_WORDS = ["aunt", "uncle", "car", "bus", "bread", "cheese", "river", "city", "music", "doctor",
               "green", "energy", "computer", "language", "translation", "galicia", "spain", "ocean",
               "summer", "winter"]

start = time.time()
slow = {w: [n for n, _ in word_vectors.most_similar(w, topn=5)] for w in QUERY_WORDS}
elapsed_loop = time.time() - start

unit_vectors = normalize_rows(word_vectors.vectors)
start = time.time()
fast = batch_neighbors(QUERY_WORDS, unit_vectors, k=5)
elapsed_batch = time.time() - start

print(f"one query at a time: {elapsed_loop * 1000:7.1f} ms")
print(f"all queries at once: {elapsed_batch * 1000:7.1f} ms")
print(f"same neighbors: {sum(slow[w] == fast[w] for w in QUERY_WORDS)}/{len(QUERY_WORDS)}")
```

</details>

## A 2-D map of related words

Real embeddings have 50 dimensions here (GPT-3-scale models use hundreds). We can't plot
50 dimensions directly, so we reduce to 2 with PCA — a standard technique that finds the 2
directions capturing the most variation, and projects everything onto just those. Expect
to see family words cluster together, and vehicle words cluster separately.

### ✏️ Exercise 4 — Add a category to the map

Add **four words of a new category** (food, animals, colors, ...) to `WORDS`, then
compute the 2-D PCA coordinates and run the cell. Does the new category form its own
cluster?

<details><summary><b>Hints</b> (click to open)</summary>

- GloVe's vocabulary is lowercase English words, e.g. `"bread"`.
- `PCA(n_components=k)` keeps `k` dimensions.

</details>

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

WORDS = ["aunt", "uncle", "mother", "father", "sister", "brother",
         "car", "bus", "train", "bicycle", "house", "apartment"]
# TODO: add four words of a new category to WORDS

vectors = ___  # TODO: one row per word in WORDS
coords = ___  # TODO: PCA down to 2 dimensions

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for word, (x, y) in zip(WORDS, coords):
    plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 5))
plt.title("50-dimensional word embeddings, projected to 2-D")
plt.show()

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert len(WORDS) >= 16, "add four words"
assert all(w in word_vectors.key_to_index for w in WORDS), [w for w in WORDS if w not in word_vectors.key_to_index]
assert coords.shape == (len(WORDS), 2)
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

WORDS = ["aunt", "uncle", "mother", "father", "sister", "brother",
         "car", "bus", "train", "bicycle", "house", "apartment",
         "bread", "cheese", "apple", "soup"]

vectors = np.array([word_vectors[w] for w in WORDS])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for word, (x, y) in zip(WORDS, coords):
    plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 5))
plt.title("50-dimensional word embeddings, projected to 2-D")
plt.show()
```

</details>

### ✏️ Exercise 5 — PCA from its definition (advanced)

PCA finds the directions along which the data varies most. With the data matrix X
(one row per word) **centered** (the mean row subtracted from every row), the singular value
decomposition X = U · S · Vᵀ gives them directly:

- the rows of Vᵀ are the directions (the principal components), most variation first;
- the 2-D coordinates are X · (the first two rows of Vᵀ)ᵀ;
- the share of the variance along component i is sᵢ² / Σⱼ sⱼ², with sᵢ the singular values.

Complete `pca_2d` with `np.linalg.svd(X, full_matrices=False)`, and check it against
scikit-learn. One detail: a direction and its opposite are equally good, so each
component may come out with the opposite sign to scikit-learn's.

<details><summary><b>Hints</b> (click to open)</summary>

- `X.mean(axis=0)` is the mean row.
- `Vt[:2]` has shape (2, 50); its transpose has shape (50, 2).
- `S ** 2 / np.sum(S ** 2)` gives the share for every component; keep the first two.

</details>

In [ ]:
def pca_2d(X):
    centered = ___  # TODO: subtract the mean row
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    coords = ___  # TODO: project onto the first two components
    ratio = ___  # TODO: explained variance ratio of the first two components
    return coords, ratio

my_coords, my_ratio = pca_2d(vectors)
sk = PCA(n_components=2).fit(vectors)
print("explained variance ratio, mine:   ", np.round(my_ratio, 4))
print("explained variance ratio, sklearn:", np.round(sk.explained_variance_ratio_, 4))

In [ ]:
# Check for Exercise 5. Run this cell as is.
_sk = PCA(n_components=2).fit_transform(vectors)
assert my_coords.shape == (len(WORDS), 2)
for _i in range(2):
    _same = np.allclose(my_coords[:, _i], _sk[:, _i], atol=1e-4)
    _flipped = np.allclose(my_coords[:, _i], -_sk[:, _i], atol=1e-4)
    assert _same or _flipped, f"component {_i} differs from scikit-learn's, even with the sign flipped"
assert np.allclose(my_ratio, sk.explained_variance_ratio_, atol=1e-5)
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
def pca_2d(X):
    centered = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    coords = centered @ Vt[:2].T
    ratio = S[:2] ** 2 / np.sum(S ** 2)
    return coords, ratio

my_coords, my_ratio = pca_2d(vectors)
sk = PCA(n_components=2).fit(vectors)
print("explained variance ratio, mine:   ", np.round(my_ratio, 4))
print("explained variance ratio, sklearn:", np.round(sk.explained_variance_ratio_, 4))
```

</details>

**Reflection:** Look at the scatter plot above. Do family words end up clustered
near each other, and vehicle words clustered separately, even though nobody told the
model what a "family" or "vehicle" category is? If so, where did that structure come
from? Connect your answer back to the skip-gram training objective — what is the model actually
being trained to predict, and how could that alone produce categories nobody explicitly
defined?

## Part B — Does Galician actually look distinct to a multilingual model?

This notebook's Galician thread asks: does a general-purpose model actually represent
Galician on its own terms, or does it default to what it has much more data for —
Spanish and Portuguese, both very close relatives? This section probes that directly,
using a small multilingual sentence-embedding model.

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

print("Downloading paraphrase-multilingual-MiniLM-L12-v2 (about 470MB)...")
multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Loaded.")

## The same words, four languages

"Tía miña" is the phrase the attention alignment example works through. Here are its two
words, plus the whole phrase, in Galician, Spanish, Portuguese, and English.

### ✏️ Exercise 6 — Split the embeddings back by language

`multi_model.encode` returns one matrix with all 12 embeddings, in the order of
`flat_texts`. Fill in the loop that hands each language its own 3 rows.

<details><summary><b>Hints</b> (click to open)</summary>

- `i` is the row where the current language starts.
- Each language has `len(texts)` rows.

</details>

In [ ]:
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time
import numpy as np
import pandas as pd

WORDS_BY_LANGUAGE = {
    "gl": ["tía", "miña", "unha tía miña"],
    "es": ["tía", "mía", "una tía mía"],
    "pt": ["tia", "minha", "uma tia minha"],
    "en": ["aunt", "my", "an aunt of mine"],
}

flat_texts = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]

tracker_embed = EmissionsTracker(
    project_name="green-embeddings-multilingual",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_embed.start()
start_time = time.time()
embeddings = multi_model.encode(flat_texts, normalize_embeddings=True)
elapsed_embed = time.time() - start_time
emissions_embed_kg = tracker_embed.stop()

vectors_by_language = {}
i = 0
for lang, texts in WORDS_BY_LANGUAGE.items():
    # TODO: take this language's rows of embeddings, then move i forward
    ___

print(f"Embedded {len(flat_texts)} texts across 4 languages in {elapsed_embed:.2f}s "
      f"({emissions_embed_kg * 1000:.6f} g CO2eq).")
print("Tiny on its own — Notebook 3 (01_baseline_measurement.ipynb) tracks costs like")
print("this at real production scale, across four different translation approaches.")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert list(vectors_by_language) == ["gl", "es", "pt", "en"]
assert all(v.shape == (3, embeddings.shape[1]) for v in vectors_by_language.values())
assert np.allclose(vectors_by_language["es"][0], embeddings[3]), "'tía' in Spanish is row 3"
assert np.allclose(vectors_by_language["en"][2], embeddings[-1])
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time
import numpy as np
import pandas as pd

WORDS_BY_LANGUAGE = {
    "gl": ["tía", "miña", "unha tía miña"],
    "es": ["tía", "mía", "una tía mía"],
    "pt": ["tia", "minha", "uma tia minha"],
    "en": ["aunt", "my", "an aunt of mine"],
}

flat_texts = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]

tracker_embed = EmissionsTracker(
    project_name="green-embeddings-multilingual",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_embed.start()
start_time = time.time()
embeddings = multi_model.encode(flat_texts, normalize_embeddings=True)
elapsed_embed = time.time() - start_time
emissions_embed_kg = tracker_embed.stop()

vectors_by_language = {}
i = 0
for lang, texts in WORDS_BY_LANGUAGE.items():
    vectors_by_language[lang] = embeddings[i : i + len(texts)]
    i += len(texts)

print(f"Embedded {len(flat_texts)} texts across 4 languages in {elapsed_embed:.2f}s "
      f"({emissions_embed_kg * 1000:.6f} g CO2eq).")
print("Tiny on its own — Notebook 3 (01_baseline_measurement.ipynb) tracks costs like")
print("this at real production scale, across four different translation approaches.")
```

</details>

## Cosine similarity: Galician against the other three

For each of the three items (the two words and the full phrase), how similar is the
Galician embedding to its Spanish, Portuguese, and English counterpart? (Since the vectors
are normalized, a plain dot product *is* the cosine similarity.)

### ✏️ Exercise 7 — Fill the similarity table

For each item, compare the Galician vector with its Spanish, Portuguese, and English
counterparts.

<details><summary><b>Hints</b> (click to open)</summary>

- The vectors were encoded with `normalize_embeddings=True`, so each has length 1.
- For vectors of length 1, the dot product is the cosine similarity.

</details>

In [ ]:
labels = ["tía / aunt", "miña / my", "full phrase"]
rows = []
for idx, label in enumerate(labels):
    gl_vec = ___  # TODO: the Galician vector for this item
    row = {"item": label}
    for lang in ["es", "pt", "en"]:
        other_vec = vectors_by_language[lang][idx]
        row[f"gl vs {lang}"] = ___  # TODO: cosine similarity (the vectors are normalized)
    rows.append(row)

similarity_df = pd.DataFrame(rows)
similarity_df

In [ ]:
# Check for Exercise 7. Run this cell as is.
assert similarity_df.shape == (3, 4)
_g, _p = vectors_by_language["gl"][0], vectors_by_language["pt"][0]
_expected = float(np.dot(_g, _p) / (np.linalg.norm(_g) * np.linalg.norm(_p)))
assert abs(similarity_df.loc[0, "gl vs pt"] - _expected) < 1e-5
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
labels = ["tía / aunt", "miña / my", "full phrase"]
rows = []
for idx, label in enumerate(labels):
    gl_vec = vectors_by_language["gl"][idx]
    row = {"item": label}
    for lang in ["es", "pt", "en"]:
        other_vec = vectors_by_language[lang][idx]
        row[f"gl vs {lang}"] = float(np.dot(gl_vec, other_vec))
    rows.append(row)

similarity_df = pd.DataFrame(rows)
similarity_df
```

</details>

## Visualizing the gap

The table above has the numbers; a bar chart makes the pattern easier to see at a
glance, and a 2-D map of the actual embeddings shows *why* the numbers come out that
way — literally where each language's version of these phrases sits in vector space.

In [ ]:
import matplotlib.pyplot as plt

similarity_df.set_index("item").plot(kind="bar", figsize=(7, 5), ylim=(0, 1))
plt.ylabel("cosine similarity to Galician")
plt.title("How close does each language sit to Galician?")
plt.xticks(rotation=0)
plt.legend(title="compared to")
plt.tight_layout()
plt.show()

## A 2-D map of the same 12 embeddings

Same PCA idea as Part A: project the real 384-dimensional embeddings down to 2
dimensions, and plot all 12 (word/phrase, language) pairs together, colored by
language.

In [ ]:
from sklearn.decomposition import PCA

point_langs = [lang for lang, texts in WORDS_BY_LANGUAGE.items() for _ in texts]
point_labels = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]
lang_names = {"gl": "Galician", "es": "Spanish", "pt": "Portuguese", "en": "English"}
colors = {"gl": "tab:blue", "es": "tab:orange", "pt": "tab:green", "en": "tab:red"}

coords = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(7, 6))
for lang in WORDS_BY_LANGUAGE:
    mask = [l == lang for l in point_langs]
    plt.scatter(coords[mask, 0], coords[mask, 1], label=lang_names[lang], color=colors[lang])
for (x, y), text in zip(coords, point_labels):
    plt.annotate(text, (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.title("Same phrases, four languages, projected to 2-D")
plt.legend()
plt.tight_layout()
plt.show()

**Reflection:** Compare the "gl vs es" numbers to the "gl vs en" numbers in the table
above, and look at where the Galician points land in the 2-D map. Which language does
Galician sit closest to? If a Galician word sits almost as close to its Spanish
translation as it sits to itself, what would that mean about how this model internally
represents Galician — is it treating Galician as its own distinct language, or folding it
into something else? (Also notice how small the measured cost of embedding these 12 short
phrases was — translating and evaluating a full sentence at realistic scale costs several
orders of magnitude more.)

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.